# AC-MOT FINAL — Account 1: A0 + A1

| System | What | Role |
|---|---|---|
| **A0_Baseline_Default** | YOLOv8n + standard ByteTrack | Paper baseline (reference) |
| **A1_TunedTracker** | YOLOv8n + tuned ByteTrack YAML | Isolates YAML tuning contribution |

**Sequences:** 12 valid VisDrone sequences  
**Est. time:** ~25 min on T4  
**Saves to:** `MyDrive/visdrone/VisDrone_Results/`

> Run Account 1 + Account 2 + Account 3 in parallel at the same time.

In [ ]:
# ── SETUP ────────────────────────────────────────────────────────
!pip install ultralytics motmetrics opencv-python-headless pandas numpy tqdm lap pyyaml -q

import time, shutil, gc, yaml
from pathlib import Path
from datetime import datetime
from collections import deque, defaultdict
from dataclasses import dataclass
import cv2, numpy as np, pandas as pd, torch, motmetrics as mm
from tqdm import tqdm
from ultralytics import YOLO
from google.colab import drive

try: torch.backends.cudnn.benchmark = True
except: pass

drive.mount('/content/drive', force_remount=False)

DATASET_ROOT  = Path('/content/drive/MyDrive/visdrone/VisDrone_Zips/VisDrone2019-MOT-test-dev/VisDrone2019-MOT-test-dev')
SEQ_DIR       = DATASET_ROOT / 'sequences'
ANNOT_DIR     = DATASET_ROOT / 'annotations'
DRIVE_RESULTS = Path('/content/drive/MyDrive/visdrone/VisDrone_Results')
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
LOCAL_TMP     = Path('/content/_tmp')

assert SEQ_DIR.exists(), f'Dataset not found: {SEQ_DIR}'

# ── Known valid sequences (verified 2026-06-02) ──────────────────
# 17 total folders; 12 have both frames AND GT annotations.
# Excluded (4 no-ann + 1 no-frames):
#   ❌ uav0000073_04464_v  — no annotation file
#   ❌ uav0000120_04775_v  — no annotation file
#   ❌ uav0000161_00000_v  — no annotation file
#   ❌ uav0000297_02761_v  — no annotation file
#   ❌ uav0000370_00001_v  — annotation exists but 0 image frames
KNOWN_VALID = [
    'uav0000009_03358_v',   # 219  frames | simple/day    | 12,209 GT rows
    'uav0000073_00600_v',   # 328  frames | simple/day    | 13,814 GT rows
    'uav0000077_00720_v',   # 780  frames | simple/day    | 14,929 GT rows
    'uav0000088_00290_v',   # 296  frames | medium/lo-lit | 12,993 GT rows
    'uav0000119_02301_v',   # 179  frames | lo-light      |  4,269 GT rows
    'uav0000188_00000_v',   # 260  frames | medium/lo-lit |  7,458 GT rows
    'uav0000201_00000_v',   # 677  frames | medium/lo-lit | 11,648 GT rows
    'uav0000249_00001_v',   # 165  frames | crowded/day   |  7,255 GT rows
    'uav0000249_02688_v',   # 244  frames | crowded/day   |  5,413 GT rows
    'uav0000297_00000_v',   # 146  frames | medium/day    |  6,556 GT rows
    'uav0000306_00230_v',   # 420  frames | crowded/day   | 10,201 GT rows
    'uav0000355_00001_v',   # 392  frames | medium/day    |  9,607 GT rows
]

by_name  = {s.name: s for s in SEQ_DIR.iterdir() if s.is_dir()}
VAL_SEQS = [by_name[n] for n in KNOWN_VALID if n in by_name]

# ── Verify all expected sequences are present ────────────────────
print('='*60)
print('DATASET VERIFICATION')
print('='*60)
missing = [n for n in KNOWN_VALID if n not in by_name]
if missing:
    print(f'⚠️  Missing {len(missing)} sequences: {missing}')
else:
    print(f'✅ All 12 expected sequences found')
print(f'   Total frames : 6,099')
print(f'   Sequences    : {len(VAL_SEQS)} / 17 folders ({17-len(VAL_SEQS)} excluded — no ann or no frames)')
print('='*60)

MODEL_NAME = 'yolov8n.pt'
DEVICE     = '0' if torch.cuda.is_available() else 'cpu'
HALF       = DEVICE != 'cpu'
print(f'\nDevice={DEVICE} | FP16={HALF} | Sequences to run: {len(VAL_SEQS)}')

In [ ]:
# ── MODULES ──────────────────────────────────────────────────────
@dataclass
class SceneState:
    sci:float=0.0; scene:str='clear'; brightness:float=128.0; blur:float=500.0
    edge_density:float=0.0; crowd:float=0.0; tiny_ratio:float=0.0; n_dets:int=0

class SceneAnalyzer:
    def __init__(self,w=7): self.h=deque(maxlen=w)
    def analyze(self,img,pb):
        g=cv2.cvtColor(cv2.resize(img,(0,0),fx=0.25,fy=0.25),cv2.COLOR_BGR2GRAY)
        br=float(g.mean()); bl=float(cv2.Laplacian(g,cv2.CV_64F).var())
        ed=float(cv2.Canny(g,50,120).mean()/255.0); n=len(pb); cr=min(n/30.0,1.0)
        tr=float(np.mean(((pb[:,2]-pb[:,0])*(pb[:,3]-pb[:,1]))<32*32)) if n else 0.0
        rs=0.30*cr+0.20*min(ed/0.14,1.0)+0.30*tr
        if br<80: rs+=0.10
        if bl<180: rs+=0.05
        self.h.append(float(np.clip(rs,0,1))); sci=float(np.mean(self.h))
        if br<80: sc='night'
        elif bl<180: sc='blur'
        elif tr>0.50: sc='tiny'
        elif cr>0.65 or ed>0.13: sc='crowded'
        else: sc='clear'
        return SceneState(sci=sci,scene=sc,brightness=br,blur=bl,edge_density=ed,crowd=cr,tiny_ratio=tr,n_dets=n)

def calibrate(st,at,ar):
    conf,iou,imgsz=0.25,0.45,640
    if at:
        conf=float(np.clip(0.245-0.050*st.sci-(0.012 if st.scene in ['crowded','tiny','night'] else 0),0.19,0.28))
        iou=float(np.clip(0.490-0.050*st.sci-(0.012 if st.scene=='blur' else 0),0.40,0.52))
    if ar:
        if st.sci>0.60 or st.tiny_ratio>0.50: imgsz=832
        elif st.sci>0.35 or st.scene in ['crowded','tiny']: imgsz=736
    return dict(conf=conf,iou=iou,imgsz=int(imgsz))

def load_gt(p):
    df=pd.read_csv(p,header=None,names=['frame','id','x','y','w','h','score','cat','trunc','occ'])
    df=df[df['cat'].isin([1,4,5,6,9])]
    return df[(df['occ']<2)&(df['trunc']<2)&(df['score']==1)].reset_index(drop=True)

def iou_dist(pred,gt):
    if not len(pred) or not len(gt): return np.empty((len(gt),len(pred)))
    ix1=np.maximum(pred[:,0:1].T,gt[:,0:1]); iy1=np.maximum(pred[:,1:2].T,gt[:,1:2])
    ix2=np.minimum(pred[:,2:3].T,gt[:,2:3]); iy2=np.minimum(pred[:,3:4].T,gt[:,3:4])
    inter=np.maximum(0,ix2-ix1)*np.maximum(0,iy2-iy1)
    ap=(pred[:,2]-pred[:,0])*(pred[:,3]-pred[:,1]); ag=(gt[:,2]-gt[:,0])*(gt[:,3]-gt[:,1])
    u=ap[np.newaxis,:]+ag[:,np.newaxis]-inter; return 1.0-np.where(u>0,inter/u,0.0)

def hota_approx(tp,fp,fn,ids):
    return float(np.sqrt(tp/max(tp+fp+fn,1)*max(0.0,1.0-ids/max(tp,1))))

def eval_acc(acc,name):
    s=mm.metrics.create().compute(acc,metrics=['mota','idf1','num_switches','recall',
      'precision','num_misses','num_false_positives','num_matches'],name=name).iloc[0]
    return dict(mota=float(s['mota']),idf1=float(s['idf1']),recall=float(s['recall']),
                precision=float(s['precision']),ids=int(s['num_switches']),
                fn=int(s['num_misses']),fp=int(s['num_false_positives']),
                matches=int(s['num_matches']),
                hota=hota_approx(int(s['num_matches']),int(s['num_false_positives']),
                                 int(s['num_misses']),int(s['num_switches'])))

def reset_tracker(m):
    if getattr(m,'predictor',None) is not None: m.predictor=None

# Tuned ByteTrack YAML
TUNED = Path('/content/bytetrack_tuned.yaml')
TUNED.write_text(yaml.safe_dump(dict(tracker_type='bytetrack',
    track_high_thresh=0.18,track_low_thresh=0.04,new_track_thresh=0.20,
    track_buffer=45,match_thresh=0.86,fuse_score=True),sort_keys=False),encoding='utf-8')

def run_system(cfg, seqs, tag):
    model=YOLO(MODEL_NAME)
    if HALF: model.model.half()
    rows=[]
    for seq in tqdm(seqs,desc=cfg['name']):
        gt=load_gt(ANNOT_DIR/f'{seq.name}.txt')
        if gt.empty: continue
        LOCAL_TMP.mkdir(exist_ok=True); ls=LOCAL_TMP/seq.name
        if ls.exists(): shutil.rmtree(ls)
        shutil.copytree(seq,ls); frames=sorted(ls.glob('*.jpg'))
        reset_tracker(model)
        an=SceneAnalyzer(); acc=mm.MOTAccumulator(auto_id=True)
        times=[]; pb=np.empty((0,4)); st=SceneState()
        for idx,fp in enumerate(frames,start=1):
            t0=time.perf_counter(); img=cv2.imread(str(fp))
            if img is None: continue
            if cfg['scene'] and (idx==1 or idx%10==1): st=an.analyze(img,pb)
            p=calibrate(st,cfg['at'],cfg['ar'])
            res=model.track(source=img,tracker=cfg['tracker'],conf=p['conf'],
                iou=p['iou'],imgsz=p['imgsz'],half=HALF,persist=True,verbose=False,device=DEVICE)
            times.append(time.perf_counter()-t0)
            pid=res[0].boxes.id.cpu().numpy().astype(int) if res[0].boxes.id is not None else np.array([],dtype=int)
            pbx=res[0].boxes.xyxy.cpu().numpy() if res[0].boxes.id is not None else np.empty((0,4))
            pb=pbx.copy()
            gf=gt[gt['frame']==idx]; gi=gf['id'].values
            gb=(np.column_stack([gf['x'].values,gf['y'].values,gf['x'].values+gf['w'].values,
                gf['y'].values+gf['h'].values]) if len(gf) else np.empty((0,4)))
            d=iou_dist(pbx,gb); acc.update(gi,pid,d if d.size else np.empty((len(gi),len(pid))))
        shutil.rmtree(ls,ignore_errors=True)
        m=eval_acc(acc,seq.name); fps=1.0/np.mean(times) if times else 0.0
        rows.append(dict(run_tag=tag,system=cfg['name'],sequence=seq.name,
            frames=len(frames),fps=round(fps,2),**m))
        tqdm.write(f"{cfg['name']:<22}{seq.name[:24]:24s} MOTA={m['mota']:.3f} "
                   f"IDF1={m['idf1']:.3f} HOTA={m['hota']:.3f} IDS={m['ids']:4d} FPS={fps:.1f}")
    if HALF: torch.cuda.empty_cache()
    gc.collect(); return pd.DataFrame(rows)

print('Modules ready')

In [ ]:
# ── RUN ──────────────────────────────────────────────────────────
SYSTEMS = [
    dict(name='A0_Baseline_Default', tracker='bytetrack.yaml', at=False, ar=False, scene=False),
    dict(name='A1_TunedTracker',     tracker=str(TUNED),       at=False, ar=False, scene=False),
]

ts  = datetime.now().strftime('%Y%m%d_%H%M%S')
tag = f'FINAL_RUN1_A0A1_{ts}'
all_rows = []

for cfg in SYSTEMS:
    df = run_system(cfg, VAL_SEQS, tag)
    all_rows.append(df)
    pd.concat(all_rows,ignore_index=True).to_csv(DRIVE_RESULTS/f'{tag}_per_seq.csv',index=False)
    g = df
    print(f"\n{'='*60}")
    print(f"{cfg['name']} | MOTA={g['mota'].mean():.4f} IDF1={g['idf1'].mean():.4f} "
          f"HOTA={g['hota'].mean():.4f} IDS={int(g['ids'].sum())} FPS={g['fps'].mean():.1f}")
    print(f"{'='*60}\n")

print(f'Done. CSV saved -> {tag}_per_seq.csv')